In [7]:
import pandas as pd

# Define the filenames
input_filename = 'recipes.csv'
output_filename = 'recipes_deduplicated.csv'

# --- 1. Load the Original Dataset ---
try:
    df = pd.read_csv(input_filename)
    print(f"Successfully loaded '{input_filename}'.\n")
except FileNotFoundError:
    print(f"Error: The file '{input_filename}' was not found.")
    print("Please make sure it's in the same folder as this notebook.")
    exit()

# --- 2. Report on Duplicates Before Removal ---
original_row_count = len(df)
# The .duplicated() method returns a series of True/False values
# .sum() counts how many 'True' values there are
num_duplicates = df.duplicated(subset=['recipe_name']).sum()

print(f"Original number of rows: {original_row_count}")
print(f"Number of duplicate recipe names found: {num_duplicates}\n")


# --- 3. Remove the Duplicates ---
# The drop_duplicates() function removes rows that are duplicates based on the 'subset' column.
# 'keep="first"' tells pandas to keep the first occurrence it finds and remove the rest.
deduplicated_df = df.drop_duplicates(subset=['recipe_name'], keep='first')


# --- 4. Report the Results ---
new_row_count = len(deduplicated_df)
print("--- Processing Complete ---")
print(f"Removed {original_row_count - new_row_count} duplicate rows.")
print(f"The new dataset has {new_row_count} unique recipes.\n")


# --- 5. Save the Cleaned File ---
deduplicated_df.to_csv(output_filename, index=False)
print(f"The deduplicated data has been saved to a new file: '{output_filename}'")


# --- 6. Display a Preview of the Clean Data ---
print("\n--- Preview of the Deduplicated Data ---")
print(deduplicated_df.head())

Successfully loaded 'recipes.csv'.

Original number of rows: 1090
Number of duplicate recipe names found: 129

--- Processing Complete ---
Removed 129 duplicate rows.
The new dataset has 961 unique recipes.

The deduplicated data has been saved to a new file: 'recipes_deduplicated.csv'

--- Preview of the Deduplicated Data ---
   Unnamed: 0                  recipe_name prep_time cook_time     total_time  \
0           0     Apple-Cranberry Crostada       NaN       NaN            NaN   
1           1    Apple Pie by Grandma Ople   30 mins     1 hrs  1 hrs 30 mins   
2           2  Sarah's Homemade Applesauce   10 mins   15 mins        25 mins   
3           3                  Apple Crisp   30 mins   45 mins  1 hrs 15 mins   
4           4            Apple Pie Filling   20 mins   20 mins  2 hrs 40 mins   

   servings              yield  \
0         8  6 to 8 - servings   
1         8       1 9-inch pie   
2         4                NaN   
3        12    1 9x13-inch pan   
4        40   

In [5]:
import pandas as pd
import re

# --- 1. Define Filenames ---
main_file = 'recipes_deduplicated.csv'
extra_file = 'recipesExtra.csv'
output_file = 'recipes_final_1000.csv'
target_rows = 1000

# --- 2. Load the Datasets ---
try:
    df_main = pd.read_csv(main_file)
    df_extra = pd.read_csv(extra_file)
    print(f"Successfully loaded '{main_file}' ({len(df_main)} rows) and '{extra_file}' ({len(df_extra)} rows).\n")
except FileNotFoundError as e:
    print(f"Error: {e}. Make sure both CSV files are in the same folder.")
    exit()

# --- 3. Clean and Standardize the 'recipesExtra.csv' DataFrame ---
print("--- Standardizing 'recipesExtra.csv' to match the main file's format ---")

# Step A: Clean up column names (lowercase, strip spaces) for consistent matching
df_extra.columns = df_extra.columns.str.strip().str.lower()

# Step B: Create a new, standardized DataFrame from the extra data
df_extra_standardized = pd.DataFrame()

# Map the corresponding columns
df_extra_standardized['recipe_name'] = df_extra['name']
df_extra_standardized['ingredients'] = df_extra['recipeingredientparts']
df_extra_standardized['directions'] = df_extra['recipeinstructions']
df_extra_standardized['servings'] = df_extra['recipeservings']
df_extra_standardized['yield'] = df_extra['recipeyield']
df_extra_standardized['rating'] = df_extra['aggregatedrating']
# Add other columns from the main file as empty for now
df_extra_standardized['url'] = ''
df_extra_standardized['cuisine_path'] = 'Unknown'
df_extra_standardized['img_src'] = ''

# Step C: Convert ISO time format (e.g., PT1H30M) to human-readable format
def convert_iso_to_human_readable(iso_string):
    if pd.isna(iso_string): return None
    try:
        hours_match = re.search(r'(\d+)H', str(iso_string))
        minutes_match = re.search(r'(\d+)M', str(iso_string))
        hours = int(hours_match.group(1)) if hours_match else 0
        minutes = int(minutes_match.group(1)) if minutes_match else 0
        total_minutes = hours * 60 + minutes
        if total_minutes == 0: return None # Treat 'PT0S' or empty PT as missing
        
        h, m = divmod(total_minutes, 60)
        parts = []
        if h > 0: parts.append(f"{h} hrs")
        if m > 0: parts.append(f"{m} mins")
        return " ".join(parts) if parts else None
    except:
        return None

df_extra_standardized['prep_time'] = df_extra['preptime'].apply(convert_iso_to_human_readable)
df_extra_standardized['cook_time'] = df_extra['cooktime'].apply(convert_iso_to_human_readable)
df_extra_standardized['total_time'] = df_extra['totaltime'].apply(convert_iso_to_human_readable)
print("Time formats converted successfully.")

# Step D: Synthesize the 'nutrition' column to match the main file's format
# This combines the separate nutrition columns into a single text string
def create_nutrition_string(row):
    parts = []
    if pd.notna(row['calories']): parts.append(f"Calories: {row['calories']}")
    if pd.notna(row['fatcontent']): parts.append(f"Fat: {row['fatcontent']}g")
    if pd.notna(row['proteincontent']): parts.append(f"Protein: {row['proteincontent']}g")
    if pd.notna(row['sugarcontent']): parts.append(f"Sugar: {row['sugarcontent']}g")
    if pd.notna(row['carbohydratecontent']): parts.append(f"Carbs: {row['carbohydratecontent']}g")
    return ", ".join(parts) if parts else 'Not Available'

df_extra_standardized['nutrition'] = df_extra.apply(create_nutrition_string, axis=1)
print("Nutrition data standardized.\n")


# --- 4. Clean and Filter Both DataFrames ---
print("--- Cleaning and Filtering Data ---")

# Clean the main DataFrame (and drop the unnecessary 'unnamed: 0' column)
df_main_cleaned = df_main.drop(columns=['unnamed: 0'], errors='ignore')
df_main_cleaned = df_main_cleaned.dropna(subset=['prep_time', 'cook_time'])
main_no_pork_mask = ~df_main_cleaned['recipe_name'].str.contains('pork', case=False, na=False) & \
                    ~df_main_cleaned['ingredients'].str.contains('pork', case=False, na=False)
df_main_cleaned = df_main_cleaned[main_no_pork_mask]
print(f"Rows remaining in main file after cleaning: {len(df_main_cleaned)}")

# Clean the extra DataFrame (which is now in the correct format)
df_extra_cleaned = df_extra_standardized.dropna(subset=['prep_time', 'cook_time', 'recipe_name', 'ingredients'])
extra_no_pork_mask = ~df_extra_cleaned['recipe_name'].str.contains('pork', case=False, na=False) & \
                     ~df_extra_cleaned['ingredients'].str.contains('pork', case=False, na=False)
df_extra_cleaned = df_extra_cleaned[extra_no_pork_mask]
print(f"Valid recipes found in extra file after cleaning: {len(df_extra_cleaned)}\n")


# --- 5. Merge to Reach Target ---
print(f"--- Merging datasets to reach {target_rows} rows ---")
rows_needed = target_rows - len(df_main_cleaned)

if rows_needed > 0:
    print(f"Need to add {rows_needed} new recipes.")
    
    # Ensure we only add unique new recipes
    existing_recipe_names = set(df_main_cleaned['recipe_name'])
    unique_new_recipes = df_extra_cleaned[~df_extra_cleaned['recipe_name'].isin(existing_recipe_names)]
    print(f"Found {len(unique_new_recipes)} unique, valid recipes to add.")
    
    if len(unique_new_recipes) >= rows_needed:
        rows_to_add = unique_new_recipes.head(rows_needed)
        final_df = pd.concat([df_main_cleaned, rows_to_add], ignore_index=True)
        print(f"\nSuccessfully added {len(rows_to_add)} new recipes.")
    else:
        final_df = pd.concat([df_main_cleaned, unique_new_recipes], ignore_index=True)
        print(f"\nWarning: Not enough unique recipes to reach {target_rows}. Added all {len(unique_new_recipes)} available.")
else:
    # If we already have 1000+ rows, just take the first 1000
    final_df = df_main_cleaned.head(target_rows).copy()
    print(f"Already have {len(df_main_cleaned)} valid recipes. Truncating to the first {target_rows}.")

# --- 6. Final Report and Save ---
final_row_count = len(final_df)
print(f"\nFinal dataset now has {final_row_count} rows.")

final_df.to_csv(output_file, index=False)
print(f"The final dataset has been saved as '{output_file}'.")

Successfully loaded 'recipes_deduplicated.csv' (961 rows) and 'recipesExtra.csv' (522517 rows).

--- Standardizing 'recipesExtra.csv' to match the main file's format ---
Time formats converted successfully.
Nutrition data standardized.

--- Cleaning and Filtering Data ---
Rows remaining in main file after cleaning: 641
Valid recipes found in extra file after cleaning: 412857

--- Merging datasets to reach 1000 rows ---
Need to add 359 new recipes.
Found 411681 unique, valid recipes to add.

Successfully added 359 new recipes.

Final dataset now has 1000 rows.
The final dataset has been saved as 'recipes_final_1000.csv'.


In [6]:
import pandas as pd
import re

# --- 1. Define Filenames ---
main_file = 'recipes_deduplicated.csv'
extra_file = 'recipesExtra.csv'
output_file = 'recipes_final_1000(2).csv'
target_rows = 1000

# --- 2. Load the Datasets ---
try:
    df_main = pd.read_csv(main_file)
    df_extra = pd.read_csv(extra_file)
    print(f"Successfully loaded '{main_file}' ({len(df_main)} rows) and '{extra_file}' ({len(df_extra)} rows).\n")
except FileNotFoundError as e:
    print(f"Error: {e}. Make sure both CSV files are in the same folder.")
    exit()

# --- 3. Define Helper Functions for Cleaning 'recipesExtra.csv' ---

def clean_c_format_string(text):
    """Removes the c("...") wrapper and extra quotes from a string."""
    if pd.isna(text):
        return None
    # Convert to string just in case it's not
    cleaned_text = str(text)
    # Check if the string starts with c(" and ends with ")
    if cleaned_text.startswith('c("') and cleaned_text.endswith('")'):
        # Get the content inside c("...")
        cleaned_text = cleaned_text[3:-2]
    # Replace the quote-comma-space sequence with just a comma and a space
    cleaned_text = cleaned_text.replace('", "', ', ')
    # Remove any remaining standalone quotes if necessary
    cleaned_text = cleaned_text.replace('"', '')
    return cleaned_text

def convert_iso_to_human_readable(iso_string):
    """Converts ISO 8601 duration format (e.g., PT1H30M) to human-readable format."""
    if pd.isna(iso_string): return None
    try:
        hours_match = re.search(r'(\d+)H', str(iso_string))
        minutes_match = re.search(r'(\d+)M', str(iso_string))
        hours = int(hours_match.group(1)) if hours_match else 0
        minutes = int(minutes_match.group(1)) if minutes_match else 0
        total_minutes = hours * 60 + minutes
        if total_minutes == 0: return None
        
        h, m = divmod(total_minutes, 60)
        parts = []
        if h > 0: parts.append(f"{h} hrs")
        if m > 0: parts.append(f"{m} mins")
        return " ".join(parts) if parts else None
    except:
        return None

def create_nutrition_string(row):
    """Combines separate nutrition columns into a single string."""
    parts = []
    if pd.notna(row.get('calories')): parts.append(f"Calories: {row['calories']}")
    if pd.notna(row.get('fatcontent')): parts.append(f"Fat: {row['fatcontent']}g")
    if pd.notna(row.get('proteincontent')): parts.append(f"Protein: {row['proteincontent']}g")
    if pd.notna(row.get('sugarcontent')): parts.append(f"Sugar: {row['sugarcontent']}g")
    if pd.notna(row.get('carbohydratecontent')): parts.append(f"Carbs: {row['carbohydratecontent']}g")
    return ", ".join(parts) if parts else 'Not Available'

# --- 4. Standardize the 'recipesExtra.csv' DataFrame ---
print("--- Standardizing 'recipesExtra.csv' to match the main file's format ---")

# Clean column names and define the mapping
df_extra.columns = df_extra.columns.str.strip().str.lower()
column_mapping = {
    'name': 'recipe_name',
    'recipeingredientparts': 'ingredients',
    'recipeinstructions': 'directions',
    'preptime': 'prep_time',
    'cooktime': 'cook_time',
    'totaltime': 'total_time',
    'recipeservings': 'servings',
    'recipeyield': 'yield',
    'aggregatedrating': 'rating'
}
df_extra.rename(columns=column_mapping, inplace=True)

# Create a new, clean DataFrame with only the columns we need
df_extra_standardized = pd.DataFrame()
df_extra_standardized['recipe_name'] = df_extra['recipe_name']

# **NEW:** Apply the cleaning function to ingredients and directions
print("Cleaning 'ingredients' and 'directions' columns...")
df_extra_standardized['ingredients'] = df_extra['ingredients'].apply(clean_c_format_string)
df_extra_standardized['directions'] = df_extra['directions'].apply(clean_c_format_string)

# Apply time conversion
print("Converting time formats...")
df_extra_standardized['prep_time'] = df_extra['prep_time'].apply(convert_iso_to_human_readable)
df_extra_standardized['cook_time'] = df_extra['cook_time'].apply(convert_iso_to_human_readable)
df_extra_standardized['total_time'] = df_extra['total_time'].apply(convert_iso_to_human_readable)

# Add and standardize other columns
df_extra_standardized['servings'] = df_extra['servings']
df_extra_standardized['yield'] = df_extra['yield']
df_extra_standardized['rating'] = df_extra['rating']
df_extra_standardized['url'] = ''  # This data is not in the extra file
df_extra_standardized['cuisine_path'] = 'Unknown'
df_extra_standardized['img_src'] = '' # This data is not in the extra file
df_extra_standardized['nutrition'] = df_extra.apply(create_nutrition_string, axis=1)

print("Standardization of 'recipesExtra.csv' complete.\n")

# --- 5. Clean and Filter Both DataFrames ---
print("--- Cleaning and Filtering Data ---")

# Clean the main DataFrame
df_main_cleaned = df_main.drop(columns=['unnamed: 0'], errors='ignore')
df_main_cleaned = df_main_cleaned.dropna(subset=['prep_time', 'cook_time', 'recipe_name', 'ingredients'])
main_no_pork_mask = ~df_main_cleaned['recipe_name'].str.contains('pork', case=False, na=False) & \
                    ~df_main_cleaned['ingredients'].str.contains('pork', case=False, na=False)
df_main_cleaned = df_main_cleaned[main_no_pork_mask]
print(f"Rows remaining in main file after cleaning: {len(df_main_cleaned)}")

# Clean the extra DataFrame
df_extra_cleaned = df_extra_standardized.dropna(subset=['prep_time', 'cook_time', 'recipe_name', 'ingredients'])
extra_no_pork_mask = ~df_extra_cleaned['recipe_name'].str.contains('pork', case=False, na=False) & \
                     ~df_extra_cleaned['ingredients'].str.contains('pork', case=False, na=False)
df_extra_cleaned = df_extra_cleaned[extra_no_pork_mask]
print(f"Valid recipes found in extra file after cleaning: {len(df_extra_cleaned)}\n")

# --- 6. Merge to Reach Target ---
print(f"--- Merging datasets to reach {target_rows} rows ---")
rows_needed = target_rows - len(df_main_cleaned)

if rows_needed > 0:
    print(f"Need to add {rows_needed} new recipes.")
    existing_recipe_names = set(df_main_cleaned['recipe_name'])
    unique_new_recipes = df_extra_cleaned[~df_extra_cleaned['recipe_name'].isin(existing_recipe_names)]
    print(f"Found {len(unique_new_recipes)} unique, valid recipes to add.")
    
    if len(unique_new_recipes) >= rows_needed:
        rows_to_add = unique_new_recipes.head(rows_needed)
        final_df = pd.concat([df_main_cleaned, rows_to_add], ignore_index=True)
        print(f"\nSuccessfully added {len(rows_to_add)} new recipes.")
    else:
        final_df = pd.concat([df_main_cleaned, unique_new_recipes], ignore_index=True)
        print(f"\nWarning: Not enough unique recipes. Added all {len(unique_new_recipes)} available.")
else:
    final_df = df_main_cleaned.head(target_rows).copy()
    print(f"Already have {len(df_main_cleaned)} valid recipes. Truncating to the first {target_rows}.")

# --- 7. Final Report and Save ---
final_row_count = len(final_df)
print(f"\nFinal dataset now has {final_row_count} rows.")

final_df.to_csv(output_file, index=False)
print(f"The final dataset has been saved as '{output_file}'.")

Successfully loaded 'recipes_deduplicated.csv' (961 rows) and 'recipesExtra.csv' (522517 rows).

--- Standardizing 'recipesExtra.csv' to match the main file's format ---
Cleaning 'ingredients' and 'directions' columns...
Converting time formats...
Standardization of 'recipesExtra.csv' complete.

--- Cleaning and Filtering Data ---
Rows remaining in main file after cleaning: 641
Valid recipes found in extra file after cleaning: 412857

--- Merging datasets to reach 1000 rows ---
Need to add 359 new recipes.
Found 411681 unique, valid recipes to add.

Successfully added 359 new recipes.

Final dataset now has 1000 rows.
The final dataset has been saved as 'recipes_final_1000(2).csv'.


In [1]:
import pandas as pd
import numpy as np

# --- 1. Define Filenames ---
input_filename = 'recipes_final_1000(2).csv'
output_filename = 'recipes_with_random_ratings.csv'

# --- 2. Load the Dataset ---
try:
    df = pd.read_csv(input_filename)
    print(f"Successfully loaded '{input_filename}'.\n")
except FileNotFoundError as e:
    print(f"Error: {e}. Make sure the CSV file is in the same folder.")
    exit()

# --- 3. Report on Missing Ratings Before Filling ---
missing_ratings_before = df['rating'].isnull().sum()
print(f"Number of rows with missing 'rating' before processing: {missing_ratings_before}")

if missing_ratings_before > 0:
    # --- 4. Fill Missing Ratings with Random Values ---
    # We will generate random numbers between 3.5 and 5.0, rounded to one decimal place.
    
    # Find the indices (row numbers) where 'rating' is missing
    missing_indices = df[df['rating'].isnull()].index
    
    # Generate the right number of random ratings
    random_ratings = np.random.uniform(low=3.5, high=5.0, size=len(missing_indices))
    random_ratings = np.round(random_ratings, 1) # Round to one decimal place
    
    # Fill the missing values using the generated random ratings
    df.loc[missing_indices, 'rating'] = random_ratings
    
    print(f"Successfully filled {len(missing_indices)} missing ratings with random values.\n")
else:
    print("No missing ratings to fill.\n")

# --- 5. Verify and Save the Final Dataset ---
missing_ratings_after = df['rating'].isnull().sum()
print(f"Number of rows with missing 'rating' after processing: {missing_ratings_after}")

# Drop the 'Unnamed: 0' column if it exists, as it's just an old index
if 'Unnamed: 0' in df.columns:
    df.drop(columns=['Unnamed: 0'], inplace=True)
    print("Removed the extra 'Unnamed: 0' column.")

# Save the final, complete DataFrame
df.to_csv(output_filename, index=False)
print(f"\nThe final dataset has been saved as '{output_filename}'.")

# --- 6. Preview the Final Data ---
print("\n--- Preview of the final dataset with filled ratings ---")
# Display a sample of rows, especially where ratings might have been missing
print(df.head(15))

Successfully loaded 'recipes_final_1000(2).csv'.

Number of rows with missing 'rating' before processing: 62
Successfully filled 62 missing ratings with random values.

Number of rows with missing 'rating' after processing: 0
Removed the extra 'Unnamed: 0' column.

The final dataset has been saved as 'recipes_with_random_ratings.csv'.

--- Preview of the final dataset with filled ratings ---
                          recipe_name prep_time cook_time     total_time  \
0           Apple Pie by Grandma Ople   30 mins     1 hrs  1 hrs 30 mins   
1         Sarah's Homemade Applesauce   10 mins   15 mins        25 mins   
2                         Apple Crisp   30 mins   45 mins  1 hrs 15 mins   
3                   Apple Pie Filling   20 mins   20 mins  2 hrs 40 mins   
4   Easy Apple Crisp with Oat Topping   20 mins   40 mins          1 hrs   
5                    Easy Apple Cider   10 mins     1 hrs  1 hrs 10 mins   
6               Apple-Cranberry Crisp   25 mins   40 mins   1 hrs 5 mins 